# TensorFlow & Keras — A Complete Beginner's Tutorial
### Tensors · Keras API · Training · Custom Layers · TPU/GPU

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)  
**Repo:** `computational-science-tutorials`

---

## What is TensorFlow?

TensorFlow is Google's open-source deep learning framework. **Keras** is the high-level API that sits on top of TensorFlow — it's how you'll write almost all your code.

**TF vs PyTorch:**
| | TensorFlow/Keras | PyTorch |
|---|---|---|
| API style | `model.compile()` / `model.fit()` | Manual training loop |
| Deployment | TF Serving, TFLite, TF.js | TorchServe, TorchScript |
| TPU support | Native (Google Cloud) | Limited |
| Research | Growing | Dominant |
| Production | Strong (Google infra) | Strong |

## What you will learn

| Section | Topics |
|---------|--------|
| 1. Tensors | `tf.Tensor`, operations, NumPy bridge |
| 2. Variables | `tf.Variable`, gradients with `GradientTape` |
| 3. Keras Sequential | Stack layers, compile, fit |
| 4. Functional API | Multi-input, skip connections, shared layers |
| 5. Custom layers | `tf.keras.layers.Layer` subclassing |
| 6. Activations & losses | Built-ins + custom |
| 7. Callbacks | EarlyStopping, ModelCheckpoint, TensorBoard |
| 8. Custom training loop | `GradientTape` — full manual control |
| 9. `tf.data` | Efficient datasets, prefetching |
| 10. Saving/loading | SavedModel, HDF5, TFLite |
| 11. GPU/TPU | Device placement, mixed precision |
| 12. Debugging | Common errors |

---
## Section 1 — TensorFlow Tensors

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
# !pip install tensorflow  (CPU)
# !pip install tensorflow[and-cuda]  (GPU — Linux)
# On macOS M1/M2: pip install tensorflow-macos tensorflow-metal

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print(f"TensorFlow version: {tf.__version__}")
print(f"Eager execution:    {tf.executing_eagerly()}")
print(f"GPUs available:     {len(tf.config.list_physical_devices('GPU'))}")
print(f"CPUs available:     {len(tf.config.list_physical_devices('CPU'))}")

# In TF2, eager execution is on by default — operations run immediately like NumPy
# (TF1 used 'build graph → run session' — you won't see that pattern anymore)

In [ ]:
# ── 1.1 Creating tensors ─────────────────────────────────────────────────────
# tf.Tensor is TF's tensor — immutable (unlike PyTorch tensors which can be modified)

# From Python list / NumPy
a = tf.constant([1.0, 2.0, 3.0])               # 1D float32
b = tf.constant([[1, 2, 3], [4, 5, 6]])        # 2D int32
c = tf.constant([True, False, True])            # bool

print("a:", a)
print("b shape:", b.shape, "dtype:", b.dtype)

# Built-in constructors
zeros   = tf.zeros([3, 4])                       # all zeros
ones    = tf.ones([2, 3])                        # all ones
rand    = tf.random.uniform([3, 3])              # uniform [0, 1)
randn   = tf.random.normal([3, 3])              # standard normal
eye     = tf.eye(4)                              # identity
rng     = tf.range(0, 10, delta=2)              # [0, 2, 4, 6, 8]
linsp   = tf.linspace(0.0, 1.0, 5)              # [0.0, 0.25, 0.5, 0.75, 1.0]

print("\nzeros:\n", zeros.numpy())
print("range:", rng.numpy())

In [ ]:
# ── 1.2 Tensor operations ─────────────────────────────────────────────────────
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[5.0, 6.0], [7.0, 8.0]])

print("a + b:\n", (a + b).numpy())
print("a * b (element-wise):\n", (a * b).numpy())
print("a @ b (matmul):\n", (a @ b).numpy())
print("tf.matmul:\n", tf.matmul(a, b).numpy())

# Reduction
print(f"\nmean: {tf.reduce_mean(a).numpy():.2f}")
print(f"sum:  {tf.reduce_sum(a).numpy():.2f}")
print(f"max:  {tf.reduce_max(a).numpy():.2f}")

# Axis reductions
print(f"sum axis=0: {tf.reduce_sum(a, axis=0).numpy()}")  # [4, 6]
print(f"sum axis=1: {tf.reduce_sum(a, axis=1).numpy()}")  # [3, 7]

In [ ]:
# ── 1.3 NumPy ↔ TensorFlow bridge ────────────────────────────────────────────
# TF tensors and NumPy arrays interoperate seamlessly in TF2

# NumPy → TensorFlow
np_arr  = np.array([1.0, 2.0, 3.0])
tf_t    = tf.constant(np_arr)           # or tf.convert_to_tensor(np_arr)
print("NumPy → TF:", tf_t.numpy())

# TF → NumPy
back    = tf_t.numpy()                  # .numpy() method
print("TF → NumPy:", back, type(back))

# Most TF functions accept NumPy arrays directly
result = tf.reduce_mean(np_arr)         # works without explicit conversion
print("TF operation on NumPy:", result.numpy())

# Shape manipulation
x = tf.constant(range(12), dtype=tf.float32)
print(f"\nreshape: {x.shape} → {tf.reshape(x, [3, 4]).shape}")
print(f"expand_dims: {x.shape} → {tf.expand_dims(x, 0).shape}")
print(f"squeeze: {tf.expand_dims(x, 0).shape} → {tf.squeeze(tf.expand_dims(x,0)).shape}")
print(f"transpose: {tf.reshape(x,[3,4]).shape} → {tf.transpose(tf.reshape(x,[3,4])).shape}")

---
## Section 2 — Variables and GradientTape

`tf.Variable` is a mutable tensor used to store model weights. `tf.GradientTape` records operations for automatic differentiation.

In [ ]:
# ── 2.1 tf.Variable ──────────────────────────────────────────────────────────
# Variables are the learnable parameters — they persist across operations

W = tf.Variable(tf.random.normal([3, 2]), name='weights')
b = tf.Variable(tf.zeros([2]),            name='bias')

print("W:\n", W.numpy())
print("b:",   b.numpy())
print("trainable:", W.trainable)     # True by default

# Modify in place
W.assign(tf.ones([3, 2]))
W.assign_add(tf.random.normal([3, 2]) * 0.01)  # W += noise
print("\nAfter assign:", W.numpy())

In [ ]:
# ── 2.2 GradientTape — automatic differentiation ──────────────────────────────
# TF records operations inside a `with tf.GradientTape()` block

# Example: compute dy/dx for y = x^2 + 2x + 1 at x=3
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x**2 + 2*x + 1

dy_dx = tape.gradient(y, x)
print(f"x = {x.numpy()}, y = {y.numpy()}")
print(f"dy/dx = {dy_dx.numpy()}  (expected: 2*3 + 2 = 8 ✓)")

# Gradient of a linear layer
W = tf.Variable(tf.random.normal([2, 3]))
b = tf.Variable(tf.zeros([3]))
x_in = tf.constant([[1.0, 2.0]])
y_true = tf.constant([[0.5, -0.5, 1.0]])

with tf.GradientTape() as tape:
    y_pred = x_in @ W + b
    loss   = tf.reduce_mean((y_pred - y_true)**2)

# Compute gradients with respect to W and b
grads = tape.gradient(loss, [W, b])
print(f"\nLoss: {loss.numpy():.4f}")
print(f"dL/dW shape: {grads[0].shape}")
print(f"dL/db shape: {grads[1].shape}")

# Manual gradient step
lr = 0.01
W.assign_sub(lr * grads[0])   # W -= lr * dL/dW
b.assign_sub(lr * grads[1])   # b -= lr * dL/db
print("Parameters updated ✓")

---
## Section 3 — Keras Sequential API

For most models: define layers → compile → fit. Three lines of training.

In [ ]:
# ── 3.1 Build a model ────────────────────────────────────────────────────────
# Generate synthetic QSAR data
np.random.seed(42)
N, n_feat = 1000, 50
X = np.random.randn(N, n_feat).astype(np.float32)
score = X[:, 0] - X[:, 1]*0.5 + X[:, 2]*0.8
y = (score > np.percentile(score, 55)).astype(np.float32)

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Sequential model: layers applied one after another
model = tf.keras.Sequential([
    # Input layer defines the shape of input
    tf.keras.layers.Input(shape=(n_feat,)),

    # Dense = fully connected layer: y = xW^T + b
    tf.keras.layers.Dense(128),
    tf.keras.layers.BatchNormalization(),   # normalise activations
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.3),           # regularisation

    tf.keras.layers.Dense(64),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(1),              # output: single logit (binary classification)
    # No activation here — binary_crossentropy expects logits in TF
])

model.summary()

In [ ]:
# ── 3.2 Compile — specify loss, optimiser, metrics ───────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),  # from_logits=True: no sigmoid in model
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='acc'),
    ]
)

# ── 3.3 Callbacks ─────────────────────────────────────────────────────────────
# Callbacks are hooks that run at the end of each epoch
callbacks = [
    # Stop training when val_auc stops improving
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=15,
        mode='max',              # maximise AUC
        restore_best_weights=True,  # restore best checkpoint automatically
        verbose=1,
    ),
    # Save the best model weights during training
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=0,
    ),
    # Reduce lr when learning plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=0,
    ),
]

# ── 3.4 Fit — three lines to train! ──────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    class_weight={0: 1.0, 1: 2.0},  # handle class imbalance: 2× weight for positives
    verbose=1,
)

In [ ]:
# ── 3.5 Evaluate and predict ──────────────────────────────────────────────────
test_loss, test_auc, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test AUC:  {test_auc:.4f}")
print(f"Test Acc:  {test_acc:.4f}")

# Predict probabilities
logits_test = model.predict(X_test[:10], verbose=0)        # raw logits
probs_test  = tf.sigmoid(logits_test).numpy().flatten()     # → probabilities
print(f"\nFirst 10 predictions:")
for prob, true in zip(probs_test, y_test[:10]):
    pred = int(prob > 0.5)
    mark = "✓" if pred == int(true) else "✗"
    print(f"  {mark} prob={prob:.3f}  pred={pred}  true={int(true)}")

In [ ]:
# ── 3.6 Plot training history ─────────────────────────────────────────────────
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(hist_df['loss'],     '#1565C0', lw=2.2, label='Train loss')
axes[0].plot(hist_df['val_loss'], '#E74C3C', lw=2.2, label='Val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.35)

axes[1].plot(hist_df['auc'],     '#27AE60', lw=2.2, label='Train AUC')
axes[1].plot(hist_df['val_auc'], '#E67E22', lw=2.2, label='Val AUC')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].set_title('AUC During Training', fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.35)

plt.suptitle('TF/Keras Training History — Synthetic QSAR', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 4 — Functional API

For **non-linear** architectures: multiple inputs/outputs, residual connections, shared layers. Use this when Sequential is not flexible enough.

In [ ]:
# ── 4.1 Residual block (skip connection) ─────────────────────────────────────
# y = F(x, W) + x  — the core idea of ResNet

def build_residual_mlp(in_features=50, hidden=128, out_features=1, n_blocks=3):
    """
    MLP with residual connections:
      Input → ProjectionLayer → [ResBlock × n_blocks] → Output

    Each ResBlock:
      x → Dense → BN → ReLU → Dense → BN → + x → ReLU
    """
    inputs = tf.keras.layers.Input(shape=(in_features,), name='features')

    # Project input to hidden dimension
    x = tf.keras.layers.Dense(hidden, use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    # Residual blocks
    for i in range(n_blocks):
        residual = x   # store input

        x = tf.keras.layers.Dense(hidden, use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        x = tf.keras.layers.Dropout(0.2)(x)

        x = tf.keras.layers.Dense(hidden, use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)

        # Skip connection: add input to output
        x = tf.keras.layers.Add()([x, residual])
        x = tf.keras.layers.Activation('relu', name=f'block_{i}_out')(x)

    # Output head
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(out_features, name='output')(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name='ResidualMLP')


res_model = build_residual_mlp(n_feat, hidden=64, n_blocks=3)
res_model.summary()
print(f"\nTotal parameters: {res_model.count_params():,}")

In [ ]:
# ── 4.2 Multi-input model (molecular descriptors + fingerprints) ──────────────
# In drug discovery: combine physicochemical descriptors + fingerprint bits

def build_multimodal_model(n_desc=10, n_fp=1024, hidden=64):
    """
    Two input streams:
      Descriptors (MW, LogP, TPSA, ...) → MLP branch
      Fingerprint bits (ECFP4)          → MLP branch
      → Concatenate → Fusion MLP → Output
    """
    # Stream 1: Physicochemical descriptors
    desc_input = tf.keras.layers.Input(shape=(n_desc,), name='descriptors')
    d = tf.keras.layers.Dense(64, activation='relu')(desc_input)
    d = tf.keras.layers.BatchNormalization()(d)
    d = tf.keras.layers.Dense(32, activation='relu')(d)

    # Stream 2: Fingerprint bits
    fp_input = tf.keras.layers.Input(shape=(n_fp,), name='fingerprints')
    f = tf.keras.layers.Dense(256, activation='relu')(fp_input)
    f = tf.keras.layers.BatchNormalization()(f)
    f = tf.keras.layers.Dropout(0.3)(f)
    f = tf.keras.layers.Dense(64, activation='relu')(f)

    # Fusion: concatenate both streams
    fused = tf.keras.layers.Concatenate(name='fusion')([d, f])
    fused = tf.keras.layers.Dense(64, activation='relu')(fused)
    fused = tf.keras.layers.Dropout(0.3)(fused)
    output = tf.keras.layers.Dense(1, name='toxicity')(fused)

    return tf.keras.Model(
        inputs=[desc_input, fp_input],
        outputs=output,
        name='MultimodalToxModel'
    )


mm_model = build_multimodal_model(n_desc=9, n_fp=2048)
mm_model.summary()

# Usage with two inputs:
desc_batch = np.random.randn(16, 9).astype(np.float32)
fp_batch   = np.random.randint(0, 2, (16, 2048)).astype(np.float32)
preds      = mm_model.predict([desc_batch, fp_batch], verbose=0)
print(f"\nMultimodal output: {preds.shape}")

---
## Section 5 — Custom Layers

In [ ]:
# ── 5.1 Custom layer subclassing ─────────────────────────────────────────────
class GatedDenseLayer(tf.keras.layers.Layer):
    """
    Gated Linear Unit (GLU): a layer where half the neurons control
    the output of the other half via a sigmoid gate.

    output = tanh(Wx + b) ⊙ σ(Vx + c)
    Useful for molecular property prediction — learns feature importance.
    """
    def __init__(self, units, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.units   = units
        self.linear  = tf.keras.layers.Dense(units)    # transforms
        self.gate    = tf.keras.layers.Dense(units)    # gates
        self.bn      = tf.keras.layers.BatchNormalization()
        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # Two parallel projections
        linear_out = tf.keras.activations.tanh(self.linear(x))
        gate_out   = tf.keras.activations.sigmoid(self.gate(x))
        # Gated output: element-wise multiply
        out = linear_out * gate_out
        out = self.bn(out, training=training)
        return self.dropout(out, training=training)

    def get_config(self):
        """Required for model serialization."""
        config = super().get_config()
        config.update({'units': self.units})
        return config


class GatedMLP(tf.keras.Model):
    """MLP using Gated Dense Layers."""
    def __init__(self, hidden=64, out=1):
        super().__init__()
        self.g1  = GatedDenseLayer(hidden, dropout=0.2, name='gate1')
        self.g2  = GatedDenseLayer(hidden // 2, dropout=0.2, name='gate2')
        self.out = tf.keras.layers.Dense(out)

    def call(self, x, training=False):
        x = self.g1(x, training=training)
        x = self.g2(x, training=training)
        return self.out(x)


gated_model = GatedMLP(hidden=64)
test_out = gated_model(np.random.randn(8, n_feat).astype(np.float32))
print(f"GatedMLP output: {test_out.shape}")
print(f"Trainable params: {gated_model.count_params():,}")

---
## Section 6 — Activations & Loss Functions

In [ ]:
# ── 6.1 Built-in activations with visualisation ───────────────────────────────
x = tf.linspace(-4.0, 4.0, 200)

activations = {
    'relu':     tf.keras.activations.relu(x),
    'gelu':     tf.keras.activations.gelu(x),
    'swish':    tf.keras.activations.swish(x),
    'sigmoid':  tf.keras.activations.sigmoid(x),
    'tanh':     tf.keras.activations.tanh(x),
    'elu':      tf.keras.activations.elu(x),
    'selu':     tf.keras.activations.selu(x),   # self-normalising
    'softplus': tf.keras.activations.softplus(x),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
colours = ['#1565C0','#E74C3C','#27AE60','#E67E22','#8E44AD','#1ABC9C','#D35400','#C0392B']

for ax, (name, y), c in zip(axes.flat, activations.items(), colours):
    ax.plot(x.numpy(), y.numpy(), color=c, lw=2.5)
    ax.axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
    ax.axvline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
    ax.set_title(name, fontsize=12, fontweight='bold', color=c)
    ax.grid(True, alpha=0.3); ax.set_ylim(-2, 4)

plt.suptitle('Activation Functions in TF/Keras', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.2 Loss functions ─────────────────────────────────────────────────────────
print("Binary classification:")
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
logits  = tf.constant([2.0, -1.0, 0.5, -2.0])
targets = tf.constant([1.0,  0.0, 1.0,  0.0])
print(f"  BCEWithLogits: {bce(targets, logits).numpy():.4f}")

# Focal loss: down-weights easy examples → focus on hard ones
# Very useful for imbalanced toxicity datasets!
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        # y_pred: logits
        p     = tf.sigmoid(y_pred)
        bce   = tf.keras.losses.binary_crossentropy(y_true, y_pred, from_logits=True)
        p_t   = y_true * p + (1 - y_true) * (1 - p)
        alpha = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        return alpha * (1 - p_t) ** self.gamma * bce

focal = FocalLoss(gamma=2.0, alpha=0.25)
print(f"  Focal loss:    {tf.reduce_mean(focal(targets, logits)).numpy():.4f}")

print("\nMulti-class classification:")
cce     = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
logits3 = tf.constant([[2.0, 0.5, -1.0], [0.3, 3.1, 0.2]])
tgts3   = tf.constant([0, 1])   # integer class indices
print(f"  SparseCCE: {cce(tgts3, logits3).numpy():.4f}")

print("\nRegression:")
y_pred = tf.constant([2.5, 0.0, 2.1, 7.8])
y_true = tf.constant([3.0, -0.5, 2.0, 7.0])
print(f"  MSE:   {tf.keras.losses.MeanSquaredError()(y_true, y_pred).numpy():.4f}")
print(f"  MAE:   {tf.keras.losses.MeanAbsoluteError()(y_true, y_pred).numpy():.4f}")
print(f"  Huber: {tf.keras.losses.Huber()(y_true, y_pred).numpy():.4f}")

---
## Section 7 — Custom Training Loop with GradientTape

For full control (like PyTorch) — when `model.fit()` is not flexible enough.

In [ ]:
# ── 7.1 Manual training loop ──────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score

# Build fresh model
manual_model = build_residual_mlp(n_feat, hidden=64, n_blocks=2)
optimiser    = tf.keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=0.01)
loss_fn      = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# tf.data pipeline (efficient)
BATCH = 64
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(1000)
            .batch(BATCH, drop_remainder=True)
            .prefetch(tf.data.AUTOTUNE))
val_ds   = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH)

@tf.function   # compile to TF graph for speed (~2-3× faster than eager)
def train_step(X_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = manual_model(X_batch, training=True)     # training=True → dropout on
        loss   = tf.reduce_mean(
            loss_fn(tf.expand_dims(y_batch, -1), logits)
        )
    # Compute and apply gradients
    grads = tape.gradient(loss, manual_model.trainable_variables)
    # Gradient clipping
    grads, _ = tf.clip_by_global_norm(grads, 1.0)
    optimiser.apply_gradients(zip(grads, manual_model.trainable_variables))
    return loss


best_val_auc = 0
patience_cnt = 0
history_manual = {'loss': [], 'val_auc': []}

for epoch in range(1, 51):
    # Training
    epoch_losses = []
    for X_b, y_b in train_ds:
        loss = train_step(X_b, y_b)
        epoch_losses.append(float(loss))

    # Validation
    val_preds, val_true = [], []
    for X_b, y_b in val_ds:
        logits = manual_model(X_b, training=False)         # training=False → dropout off
        probs  = tf.sigmoid(logits).numpy().flatten()
        val_preds.extend(probs)
        val_true.extend(y_b.numpy())

    val_auc = roc_auc_score(val_true, val_preds)
    epoch_loss = np.mean(epoch_losses)
    history_manual['loss'].append(epoch_loss)
    history_manual['val_auc'].append(val_auc)

    # Early stopping
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        manual_model.save_weights('manual_best.weights.h5')
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss={epoch_loss:.4f} | val_AUC={val_auc:.4f}")

    if patience_cnt >= 12:
        print(f"Early stopping at epoch {epoch}")
        break

# Restore best
manual_model.load_weights('manual_best.weights.h5')
test_preds_raw = manual_model.predict(X_test, verbose=0)
test_probs     = tf.sigmoid(test_preds_raw).numpy().flatten()
test_auc_manual = roc_auc_score(y_test, test_probs)
print(f"\nManual loop Test AUC: {test_auc_manual:.4f}")

---
## Section 8 — `tf.data`: Efficient Data Pipelines

In [ ]:
# ── 8.1 tf.data pipeline patterns ────────────────────────────────────────────
# tf.data is TF's high-performance input pipeline — much faster than feeding
# NumPy arrays directly for large datasets

# From numpy arrays (most common for tabular/fingerprint data)
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))

# The canonical pipeline:
train_pipeline = (
    dataset
    .shuffle(buffer_size=len(X_train), seed=42)  # shuffle
    .batch(64, drop_remainder=True)               # batch
    .prefetch(tf.data.AUTOTUNE)                   # preload next batch while GPU trains
)

# Inspect
for X_b, y_b in train_pipeline.take(1):
    print(f"Batch X: {X_b.shape}  y: {y_b.shape}")

# From a Python generator (for large/streaming data)
def molecule_generator():
    for i in range(100):
        yield np.random.randn(n_feat).astype(np.float32), np.float32(i % 2)

gen_dataset = tf.data.Dataset.from_generator(
    molecule_generator,
    output_signature=(
        tf.TensorSpec(shape=(n_feat,), dtype=tf.float32),
        tf.TensorSpec(shape=(),       dtype=tf.float32),
    )
).batch(16)

for X_b, y_b in gen_dataset.take(1):
    print(f"Generator batch X: {X_b.shape}  y: {y_b.shape}")

# Data augmentation with .map()
def add_noise(X, y):
    """Add Gaussian noise to features (data augmentation for descriptors)."""
    noise = tf.random.normal(tf.shape(X), stddev=0.01)
    return X + noise, y

aug_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .map(add_noise, num_parallel_calls=tf.data.AUTOTUNE)  # apply on-the-fly
    .shuffle(1000)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)
print("Augmented pipeline ready ✓")

---
## Section 9 — Saving & Loading

In [ ]:
# ── 9.1 Saving formats ────────────────────────────────────────────────────────

# FORMAT 1: SavedModel (recommended, TF native, deployment-ready)
model.export('saved_model_dir')
# Load: loaded = tf.saved_model.load('saved_model_dir')
# Or:   loaded = tf.keras.models.load_model('saved_model_dir')

# FORMAT 2: Keras format (.keras) — clean, includes everything
model.save('model.keras')
loaded_keras = tf.keras.models.load_model('model.keras')
print("Keras format loaded ✓")

# FORMAT 3: Weights only (.h5 or .weights.h5)
model.save_weights('weights.weights.h5')
new_model = build_residual_mlp(n_feat)   # same architecture
new_model.build(input_shape=(None, n_feat))
new_model.load_weights('weights.weights.h5')
print("Weights loaded ✓")

# FORMAT 4: TFLite — for mobile/edge deployment
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f"TFLite model saved ({len(tflite_model)/1024:.1f} KB) ✓")

# Load and run TFLite
interp = tf.lite.Interpreter('model.tflite')
interp.allocate_tensors()
inp  = interp.get_input_details()
out  = interp.get_output_details()
interp.set_tensor(inp[0]['index'], X_test[:1])
interp.invoke()
pred = interp.get_tensor(out[0]['index'])
print(f"TFLite inference: {pred}")

---
## Section 10 — GPU & Mixed Precision

In [ ]:
# ── 10.1 GPU device placement ─────────────────────────────────────────────────
# TF auto-places on GPU when available — no need for .to(device)

print("Available devices:")
for d in tf.config.list_physical_devices():
    print(f"  {d}")

# Check where an operation runs
a = tf.constant([1.0, 2.0])
print(f"\nTensor device: {a.device}")

# Explicit device placement
with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(b, b)
    print(f"MatMul on: {c.device}")

# Multi-GPU strategy
# strategy = tf.distribute.MirroredStrategy()
# with strategy.scope():
#     model = build_model()
#     model.compile(...)
# model.fit(...)  # automatically splits batches across GPUs

# ── 10.2 Mixed precision (float16 on GPU = 2× speed) ─────────────────────────
# tf.keras.mixed_precision.set_global_policy('mixed_float16')
# model = build_model()   # weights float16, reductions float32
# model.compile(optimiser=tf.keras.optimizers.Adam(1e-3))  # optimiser handles scaling
# Restore to float32:
# tf.keras.mixed_precision.set_global_policy('float32')

current_policy = tf.keras.mixed_precision.global_policy()
print(f"\nCurrent precision policy: {current_policy.name}")

---
## Section 11 — Common Errors & Comparison with PyTorch

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║          TensorFlow/Keras Common Errors & Fixes                         ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 1: Shapes are incompatible: [16] vs [16,1]                        ║
║   Cause:  Dense(1) output is [batch,1] but targets are [batch]          ║
║   Fix:    tf.squeeze(logits, -1)  or  y_true = tf.expand_dims(y,-1)     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 2: training=True/False matters for BatchNorm and Dropout          ║
║   model(x)              → training=False (inference mode)               ║
║   model(x, training=True) → training=True  (training mode)             ║
║   Fix: Always pass training=True inside tf.GradientTape block           ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 3: ValueError: No gradients provided for any variable             ║
║   Cause:  Variable not used inside the tape block                       ║
║   Fix:    All forward pass must happen inside with tape:                ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 4: Model not learning with BinaryCrossentropy                     ║
║   Cause:  Using from_logits=False but feeding raw logits (common!)      ║
║   Fix:    Use from_logits=True and do NOT apply sigmoid in model        ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ERROR 5: GradientTape records nothing (None gradients)                  ║
║   Cause:  Using tf.Variable but modifying with .numpy() (exits TF)     ║
║   Fix:    All operations inside tape must use TF functions, not NumPy   ║
╠══════════════════════════════════════════════════════════════════════════╣
║ TF/KERAS vs PYTORCH — KEY DIFFERENCES                                   ║
║ TF:  model.compile() + model.fit()  (high-level, fast to write)         ║
║ PT:  manual: zero_grad → forward → backward → step  (more control)     ║
║                                                                          ║
║ TF:  training=True/False passed to call()                               ║
║ PT:  model.train() / model.eval() sets mode globally                   ║
║                                                                          ║
║ TF:  from_logits=True in loss (no sigmoid in model)                    ║
║ PT:  BCEWithLogitsLoss (same idea)                                      ║
║                                                                          ║
║ TF:  automatic GPU placement                                             ║
║ PT:  manual .to(device) required                                        ║
║                                                                          ║
║ CHECKLIST                                                                ║
║  □ from_logits=True in BinaryCrossentropy                               ║
║  □ No sigmoid in model when using BinaryCrossentropy(from_logits=True)  ║
║  □ model(x, training=True) inside GradientTape                         ║
║  □ model(x, training=False) / model.predict() for evaluation            ║
║  □ tape.gradient returns None if variable not in computation             ║
║  □ restore_best_weights=True in EarlyStopping                           ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---

## What to learn next

| Topic | Resource |
|---|---|
| GNN for molecules (PyTorch) | `gnn_toxicology_tutorial.ipynb` in this repo |
| TF official tutorials | tensorflow.org/tutorials |
| Keras guides | keras.io/guides |
| TF Probability | tfp.distributions — Bayesian models |
| TF Datasets | tensorflow.org/datasets |
| Deep Learning book | d2l.ai (free, has TF section) |

*Built by Himanshu Goel · [hgoelgithub.github.io](https://hgoelgithub.github.io)*